
# Compare size-exclusion cutoffs for vesicles and lipoproteins

This example demonstrates how to simulate flow cytometry data for extracellular vesicles (EVs) and lipoproteins (LPs) under different Size Exclusion Chromatography (SEC) scenarios. The simulation includes the generation of scatterer populations, opto-electronic configurations, and digital processing to analyze the resulting data.


In [ ]:
import numpy as np
import pandas as pd
from typing import Optional
from TypedUnit import ureg, AnyUnit
import seaborn as sns
import matplotlib.pyplot as plt


from FlowCyPy import fluidics
from FlowCyPy import opto_electronics
from FlowCyPy import digital_processing
from FlowCyPy import FlowCytometer

# ----------------------------
# Data geration flow cytometer
# ----------------------------

## Define the SEC simulation
Build a reusable acquisition for a specified size cutoff and particle mixture.



In [ ]:
def run_sec_swarm_demo(
    *,
    minimum_diameter_after_sec: Optional[AnyUnit],
    ev_spread: float,
    lp_spread: float,
    wavelength: float,
    threshold: str,
    trigger_channel: str,
    saturation_levels,
    ev_characteristic_diameter: AnyUnit,
    lp_characteristic_diameter: AnyUnit,
    lp_concentration: AnyUnit,
    ev_concentration: AnyUnit,
    lp_refractive_index_mean: AnyUnit,
    lp_refractive_index_standard_deviation: AnyUnit,
    ev_refractive_index_mean: AnyUnit,
    ev_refractive_index_standard_deviation: AnyUnit,
    gamma_monte_carlo_samples: int,
    ev_maximum_diameter: Optional[AnyUnit],
    lp_maximum_diameter: Optional[AnyUnit],
    reduce_concentration_consistently: bool,
    dilution_factor: float,
    run_time: AnyUnit,
):

    # 1. Fluidics Setup
    # Configure the flow cell dimensions and sample/sheath flow rates.
    flow_cell = fluidics.FlowCell(
        sample_volume_flow=fluidics.SampleFlowRate.MEDIUM.value,
        sheath_volume_flow=fluidics.SheathFlowRate.MEDIUM.value,
        width=200 * ureg.micrometer, #can be changed to fit an experimental setup
        height=100 * ureg.micrometer, #^^
        perfectly_aligned=True,
    )

    # Initialize a collection to hold different scatterer populations.
    scatterer_collection = fluidics.ScattererCollection()

    # Define the refractive index of the medium (e.g., water).
    medium_refractive_index = fluidics.distributions.Delta(1.33 * ureg.RIU)

    # 2. Lipoprotein (LP) Population Construction
    # Define the diameter distribution for Lipoproteins using Rosin-Rammler model.
    # Characterize the size uniformity and diameter spread of particles
    lp_diameter_distribution = fluidics.distributions.RosinRammler(
        scale=lp_characteristic_diameter,
        shape=float(lp_spread),
        low_cutoff=minimum_diameter_after_sec,
        high_cutoff=lp_maximum_diameter,
    )
    # Define the refractive index distribution for Lipoproteins as a Normal distribution.
    lp_refractive_index_distribution = fluidics.distributions.Normal(
        mean=lp_refractive_index_mean, # Corrected to use LP_refractive_index_mean
        standard_deviation=lp_refractive_index_standard_deviation,
        low_cutoff=None, # can be used to not incude lp particles that would overlap with evs
        high_cutoff=None, #can be used if data generated gives few very high values which are not wanted
    )
    # Calculate the proportion of LPs that are kept within the defined cutoffs.
    lp_kept_joint = lp_diameter_distribution.proportion_within_cutoffs() * lp_refractive_index_distribution.proportion_within_cutoffs()
    # Create the LP population with its properties and sampling method.
    lp_sampling_method = fluidics.populations.GammaModel(number_of_samples=gamma_monte_carlo_samples)
    lp_population = fluidics.populations.SpherePopulation(
        name="LPs",
        medium_refractive_index=medium_refractive_index,
        concentration=lp_concentration * lp_kept_joint,
        diameter=lp_diameter_distribution,
        refractive_index=lp_refractive_index_distribution,
        sampling_method=lp_sampling_method, # uses monte carlo sampling as decrease in runtime is more important than accuracy for lps
    )

    # 3. Extracellular Vesicle (EV) Population Construction
    # Define the diameter distribution for Extracellular Vesicles using Rosin-Rammler model.
    ev_diameter_distribution = fluidics.distributions.RosinRammler(
        scale=ev_characteristic_diameter,
        shape=float(ev_spread),
        low_cutoff=minimum_diameter_after_sec,
        high_cutoff=ev_maximum_diameter,
    )
    # Define the refractive index distribution for Extracellular Vesicles as a Normal distribution.
    ev_refractive_index_distribution = fluidics.distributions.Normal(
        mean=ev_refractive_index_mean,
        standard_deviation=ev_refractive_index_standard_deviation,
        low_cutoff=None, #can be used if data generated gives few very low values which are not wanted
        high_cutoff=None, # can be used to not include evs that would overlap with lps
    )
    # Calculate the proportion of EVs that are kept within the defined cutoffs.
    ev_kept_joint = ev_diameter_distribution.proportion_within_cutoffs() * ev_refractive_index_distribution.proportion_within_cutoffs()
    # Create the EV population with its properties and sampling method.
    ev_sampling_method = fluidics.populations.ExplicitModel()
    ev_population = fluidics.populations.SpherePopulation(
        name="EVs",
        medium_refractive_index=medium_refractive_index,
        concentration=ev_concentration * ev_kept_joint,
        diameter=ev_diameter_distribution,
        refractive_index=ev_refractive_index_distribution,
        sampling_method=ev_sampling_method, # uses explicit model to increase accuracy as there are fewer evs than lps
    )

    # Add populations to the scatterer collection if they exist and have non-zero concentration.
    if ev_population is not None and ev_concentration != 0:
        scatterer_collection.add_population(ev_population)
    if lp_population is not None and lp_concentration != 0:
        scatterer_collection.add_population(lp_population)

    # Apply dilution to the scatterer collection if specified.
    if dilution_factor != 1.0:
        scatterer_collection.dilute(factor=float(dilution_factor))

    # Instantiate the Fluidics module with the scatterer collection and flow cell.
    _fluidics = fluidics.Fluidics(scatterer_collection=scatterer_collection, flow_cell=flow_cell)


    # 4. Opto-Electronics Setup
    # Configure the laser beam source with Gaussian properties.
    beam = opto_electronics.source.Gaussian(
        waist_z=10 * ureg.micrometer,
        waist_y=60 * ureg.micrometer,
        wavelength=wavelength,
        optical_power=200 * ureg.milliwatt,
        include_shot_noise=False, #this is set to false to make sure the results are easy to interpret
    )

    # Configure the side scatter detector.
    detector_side = opto_electronics.Detector(
        name="side",
        phi_angle=90 * ureg.degree,
        numerical_aperture=0.9 * ureg.AU,
        responsivity=10 * ureg.ampere / ureg.watt,
        dark_current=20 * ureg.nanoampere,
    )

    # Configure the forward scatter detector.
    detector_forward = opto_electronics.Detector(
        name="forward",
        phi_angle=0 * ureg.degree,
        numerical_aperture=0.2 * ureg.AU,
        cache_numerical_aperture=0.08 * ureg.AU,
        responsivity=10 * ureg.ampere / ureg.watt,
        dark_current=20 * ureg.nanoampere,
    )

    # Configure the amplifier for signal processing.
    amplifier = opto_electronics.Amplifier(
        gain=0.1 * ureg.volt / ureg.ampere, #can be ajusted if there is no difference between signal and background noise
        bandwidth=10 * ureg.megahertz,
        voltage_noise_density=0.00001 * ureg.nanovolt / ureg.sqrt_hertz,
    )

    # Configure the digitizer for converting analog signals to digital.
    digitizer = opto_electronics.Digitizer(
        bit_depth=20,
        use_auto_range=True,
        sampling_rate=30 * ureg.megahertz,
    )

    # Define analog processing filters.
    low_pass_filter = opto_electronics.circuits.BesselLowPass(
        cutoff_frequency=2 * ureg.megahertz,
        order=4,
        gain=2,
    )
    analog_processing = [low_pass_filter]

    # Instantiate the OptoElectronics module with detectors, source, amplifier, digitizer, and analog processing.
    _opto_electronics = opto_electronics.OptoElectronics(
        detectors=[detector_side, detector_forward],
        source=beam,
        amplifier=amplifier,
        digitizer=digitizer,
        analog_processing=analog_processing,
    )

    # 5. Digital Processing Setup
    # Configure the discriminator for event detection based on a trigger channel and threshold.
    discri = digital_processing.discriminator.DynamicWindow(
        trigger_channel=trigger_channel,
        threshold=threshold,
        pre_buffer=20,
        post_buffer=20,
        max_triggers=-1,
    )

    # Configure the peak finding algorithm.
    peak_algo = digital_processing.peak_locator.GlobalPeakLocator(compute_width=False)

    # Instantiate the DigitalProcessing module with the discriminator and peak algorithm.
    _digital_processing = digital_processing.DigitalProcessing(
        discriminator=discri,
        peak_locator=peak_algo,
    )

    # 6. Flow Cytometer Run
    # Instantiate the FlowCytometer with the fluidics and background power.
    cytometer = FlowCytometer(
        fluidics=_fluidics,
        background_power=0.00001 * ureg.milliwatt,
    )

    # Run the simulation, combining opto-electronics, run time, and digital processing.
    run_record = cytometer.run(
        opto_electronics=_opto_electronics,
        run_time=run_time,
        digital_processing=_digital_processing,
    )


    # Return the record of the simulation run.
    return run_record

#---------------------------------
# Generate the populations
#---------------------------------

# Define arrays for the simulation parameters that will change across runs.
# These values correspond to the different SEC (Size Exclusion Chromatography) scenarios:
# 0 nm: EV Only, EV + LP
# 35 nm: SEC @35nm
# 70 nm: SEC @70nm

## Run the separation scenarios
Compare EVs alone, the EV and lipoprotein mixture, and two SEC size cutoffs.



In [ ]:
sec_values = [0, 0, 35, 70] * ureg.nanometer
ev_concentrations = [5e15, 5e15, 5e15, 5e15] * ureg.particle / ureg.milliliter
lp_concentrations = [0, 5e16, 5e16, 5e16] * ureg.particle / ureg.milliliter

# List to store the results (run_record) from each simulation run.
run_records = []

# Loop through the defined SEC scenarios to run the simulation for each.
# 'sec' represents the minimum diameter after SEC, 'lp_concentration' and 'ev_concentration'
# are adjusted for each specific scenario.
for index, (sec, lp_concentration, ev_concentration) in enumerate(zip(sec_values, lp_concentrations, ev_concentrations)):

    # Print current progress to track the simulation runs.
    print(f"{index} / {len(sec_values)}")

    # Initialize current_lp_min_ri for potential future refractive index cutoffs.
    current_lp_min_ri = None

    # Run the flow cytometer simulation with specific parameters for the current scenario.
    run_record = run_sec_swarm_demo(
        wavelength=488 * ureg.nanometer, # Laser wavelength
        trigger_channel='side', # Detector channel used for triggering events
        minimum_diameter_after_sec=sec, # SEC cutoff diameter for particles
        ev_maximum_diameter=1000 * ureg.nanometer, # Maximum EV diameter for sampling
        lp_maximum_diameter=1000 * ureg.nanometer, # Maximum LP diameter for sampling
        ev_spread=1.1, # Rosin-Rammler spread parameter for EVs
        lp_spread=0.9, # Rosin-Rammler spread parameter for LPs
        ev_characteristic_diameter=120 * ureg.nanometer, # Characteristic diameter for EVs
        lp_characteristic_diameter=40 * ureg.nanometer, # Characteristic diameter for LPs
        lp_concentration=lp_concentration, # Lipoprotein concentration for current run
        ev_concentration=ev_concentration, # Extracellular Vesicle concentration for current run
        dilution_factor=30_000_000, # Dilution factor applied to the sample
        threshold='5sigma', # Trigger threshold based on noise standard deviation
        gamma_monte_carlo_samples=5_000, # Number of Monte Carlo samples for gamma model
        lp_refractive_index_mean=1.47 * ureg.RIU, # Mean refractive index for LPs
        lp_refractive_index_standard_deviation=0.01 * ureg.RIU, # Std dev of refractive index for LPs
        ev_refractive_index_mean=1.39 * ureg.RIU, # Mean refractive index for EVs
        ev_refractive_index_standard_deviation=0.01 * ureg.RIU, # Std dev of refractive index for EVs
        run_time=10 * ureg.millisecond, # Total simulation run time
        saturation_levels=None, # No saturation levels applied in this simulation
        reduce_concentration_consistently=True # Explicitly passing this as its default is now removed
    )

    # Add the run record (simulation results) to the list.
    run_records.append(run_record)


# Define common variables

## Collect peak features
Combine the detected peak heights and retain the scenario labels for comparison.



In [ ]:
x_detector = "forward"
y_detector = "side"
titles_map = ['EV Only', 'EV + LP', 'SEC @35nm', 'SEC @70nm']

x_col = f"{x_detector} Height"
y_col = f"{y_detector} Height"

all_peaks_data = []

for i, record in enumerate(run_records):
    peaks_data = record.peaks
    title = titles_map[i]

    if peaks_data is not None:
        if (
            x_detector in peaks_data.index.get_level_values(0) and
            y_detector in peaks_data.index.get_level_values(0)
        ):

            df_for_plot = pd.DataFrame({
                x_col: np.asarray(peaks_data.loc[x_detector, "Height"], float),
                y_col: np.asarray(peaks_data.loc[y_detector, "Height"], float)
            }).dropna()

            df_for_plot = df_for_plot[(df_for_plot[x_col] > 0) & (df_for_plot[y_col] > 0)]

            sampled_df = df_for_plot

            sampled_df['Run_Title'] = title
            all_peaks_data.append(sampled_df)

    else:
        pass # peaks_data is None

if all_peaks_data:
    combined_df = pd.concat(all_peaks_data, ignore_index=True)
else:
    combined_df = pd.DataFrame() # Ensure combined_df is defined even if no peaks data

# Plotting part

## Compare forward and side scatter
Each panel shows the detected peak heights for one separation scenario.



In [ ]:
if not combined_df.empty:
    figure, axes = plt.subplots(2, 2, sharex=True, sharey=True, layout="constrained")
    for axis, title in zip(axes.flat, titles_map):
        data = combined_df[combined_df["Run_Title"] == title]
        axis.scatter(data[x_col], data[y_col])
        axis.set(title=title, xscale="log", yscale="log", xlim=(1, 100000), ylim=(1, 20000))
    figure.supxlabel(x_col)
    figure.supylabel(y_col)
    figure.savefig("measurements_scatter.png")
    plt.show()

# --------------------------------
# plotting
# --------------------------------

titles_map = ['EV Only', 'EV + LP', 'SEC @35nm', 'SEC @70nm']

all_event_data = []

for i, record in enumerate(run_records):
    df = record.event_collection.get_concatenated_dataframe().reset_index().rename(columns={'RefractiveIndex': 'Refractive Index'})
    df['Run_Title'] = titles_map[i]
    all_event_data.append(df)

combined_df = pd.concat(all_event_data, ignore_index=True)

## Inspect particle size and scatter
Filter the diameter range, then compare the joint distributions and their marginals.



In [ ]:
min_expected_diameter_nm = 1.0
max_expected_diameter_nm = 1000.0

combined_df['Diameter_nm'] = combined_df['Diameter'].to('nanometer').magnitude

combined_df = combined_df[
    (combined_df['Diameter_nm'] >= min_expected_diameter_nm) &
    (combined_df['Diameter_nm'] <= max_expected_diameter_nm)
].copy()

figure = plt.figure(layout="constrained")
for panel, title in zip(figure.subfigures(2, 2).flat, titles_map):
    data = combined_df[combined_df["Run_Title"] == title]
    axes = panel.subplot_mosaic(
        [["x", "."], ["joint", "y"]], width_ratios=[4, 1], height_ratios=[1, 4]
    )
    joint = axes["joint"]
    axes["x"].sharex(joint)
    axes["y"].sharey(joint)
    sns.scatterplot(data=data, x="Diameter_nm", y="Refractive Index", hue="Population", ax=joint)
    for _, population in data.groupby("Population", sort=False):
        sns.kdeplot(data=population, x="Diameter_nm", ax=axes["x"])
        sns.kdeplot(data=population, y="Refractive Index", ax=axes["y"])
    joint.set(
        xlabel="Diameter [nm]", xlim=(0, max_expected_diameter_nm * 1.1),
        ylim=(combined_df["Refractive Index"].min() * 0.99, combined_df["Refractive Index"].max() * 1.01),
    )
    axes["x"].set_axis_off()
    axes["y"].set_axis_off()
    panel.suptitle(title)
figure.savefig("scatterers_with_marginals.png")
plt.show()